# Workshop: real-gold 4D-STEM — browse, BF, DF, DPC, and 3 direct-ptycho kernels (Colab T4)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/bobleesj/a05a90185c6cddbb331342cae6d7e9c1/berk_workshop_v1.ipynb)

ONE notebook. Real gold from Hugging Face → load → browse → bright field → dark
field → DPC (via `CenterOfMassOriginModel`) → three single-shot phase-retrieval
kernels (parallax, SSB, ICOM) → side-by-side comparison.

Everything on torch on the Colab T4. Two installs only — `quantem.widget`
(TestPyPI prerelease) + `quantem` (`berk-workshop` branch). No `quantem.live`.

**Workshop punchline:** phase retrieval (parallax, SSB) recovers atomic-lattice
contrast that BF/DF physically cannot, at the same dose.

In [ ]:
!pip install -q --pre --extra-index-url https://test.pypi.org/simple/ quantem.widget huggingface_hub
!pip install -q git+https://github.com/bobleesj/quantem.git@berk-workshop

In [ ]:
import quantem as em
import quantem.widget
import torch

# cuDNN grid_sample bug at these detector dims; disable for DirectPtycho path.
torch.backends.cudnn.enabled = False

print("quantem        ", em.__version__)
print("quantem.widget ", quantem.widget.__version__)
print("torch          ", torch.__version__, "(cuDNN disabled)")
print("cuda available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU)")

In [ ]:
import os, json
import numpy as np
from huggingface_hub import snapshot_download

folder = snapshot_download("bobleesj/quantem-data", repo_type="dataset",
                           allow_patterns=["4dstem/gold_512_npy_bin8/*"])
asset = os.path.join(folder, "4dstem", "gold_512_npy_bin8")
data = np.ascontiguousarray(np.load(os.path.join(asset, "data.npy")).astype(np.float32))
meta = json.load(open(os.path.join(asset, "meta.json")))

# numpy-backed for upstream CoM + DirectPtychography (they read dataset.array)
dset = em.core.datastructures.Dataset4dstem.from_array(
    data, sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
print(f"dataset: shape {dset.shape}, dtype {dset.array.dtype}")
print(f"sampling {meta['sampling']} {meta['units']}")
print(f"optics: {meta['voltage_kV']} kV, probe {meta['probe_semiangle_mrad']} mrad, CL {meta['camera_length_mm']} mm")

## Step 1 — Browse the 4D-STEM dataset interactively

Drag the scan cursor; CBED updates live. Real-time per-scan-position BF/DF.

In [ ]:
# Torch-backed view for Show4DSTEM (GPU-fast cursor drag)
dset_torch = em.core.datastructures.Dataset4dstem.from_tensor(
    torch.from_numpy(data).to("cuda" if torch.cuda.is_available() else "cpu"),
    sampling=meta["sampling"], units=meta["units"], name=meta["name"],
)
quantem.widget.Show4DSTEM(dset_torch)

## Step 2 — Bright field (BF)

Aperture mask at the detector center; per-scan-position sum INSIDE the disk.
Inline torch on the GPU, one reduction. (`Show2D` shown alone so contrast is
not yoked to anything else.)

In [ ]:
data_f = torch.from_numpy(data).to("cuda" if torch.cuda.is_available() else "cpu")

H, W = data_f.shape[-2:]
cy, cx = H / 2, W / 2                                    # hardcoded geometric center
row = torch.arange(H, device=data_f.device, dtype=torch.float32)[:, None]
col = torch.arange(W, device=data_f.device, dtype=torch.float32)[None, :]
rr, cc = torch.meshgrid(row.squeeze(), col.squeeze(), indexing="ij")
r_from_center = ((rr - cy) ** 2 + (cc - cx) ** 2).sqrt()

BF_RADIUS_PX = 6.0
bf_mask = (r_from_center <= BF_RADIUS_PX).float()
df_mask = 1.0 - bf_mask                                  # reused in next step

bf = (data_f * bf_mask).sum(dim=(-2, -1)).cpu().numpy()
print(f"BF range [{bf.min():.1f}, {bf.max():.1f}]")

quantem.widget.Show2D(
    bf, title="Bright field",
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="gray",
)

## Step 3 — Dark field (DF)

Same data, complementary aperture mask — sum OUTSIDE the BF disk. Its own
`Show2D` widget so the contrast scale is independent of BF.

In [ ]:
df = (data_f * df_mask).sum(dim=(-2, -1)).cpu().numpy()
print(f"DF range [{df.min():.1f}, {df.max():.1f}]")

quantem.widget.Show2D(
    df, title="Dark field",
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="gray",
)

## Step 4 — DPC via `CenterOfMassOriginModel` (upstream torch on GPU)

Per-scan-position centroid (CoM) — torch on the GPU through quantem's
`CenterOfMassOriginModel`. Returns a flat `(num_dps, 2)` tensor; reshape to
`(scan_row, scan_col, 2)` for image display.

In [ ]:
from quantem.diffractive_imaging import CenterOfMassOriginModel

com_model = CenterOfMassOriginModel.from_dataset(dset, device="cuda" if torch.cuda.is_available() else "cpu")
com_model.calculate_origin()

scan_r, scan_c = dset.shape[:2]
com_map = com_model.origin_measured.view(scan_r, scan_c, 2)

# Detrend so the divergent colormap is zero-centered on signed deflection.
com_row = com_map[..., 0] - com_map[..., 0].mean()
com_col = com_map[..., 1] - com_map[..., 1].mean()
com_mag = (com_row ** 2 + com_col ** 2).sqrt()

print(f"CoM row range [{com_row.min().item():.4f}, {com_row.max().item():.4f}] px")
print(f"CoM col range [{com_col.min().item():.4f}, {com_col.max().item():.4f}] px")
print(f"|CoM| max     {com_mag.max().item():.4f} px")

quantem.widget.Show2D(
    [com_row.cpu().numpy(), com_col.cpu().numpy(), com_mag.cpu().numpy()],
    labels=["CoM row (qx)", "CoM col (qy)", "|CoM| total"],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="RdBu_r",
    link_contrast=False,
)

## Step 5 — Phase retrieval: `DirectPtychography` with three kernels

Build once, sweep three deconvolution kernels:

- **`parallax`** — parallax / tilt approximation
- **`ssb`** — single-sideband (a.k.a. aberration-corrected bright field)
- **`icom`** — integrated CoM

Two important workshop knobs:

1. **`override_aberration_coefs`** — pass the operator's calibrated `C10`,
   `C12`, `phi12` (from the gold calibration file). Without them, SSB silently
   returns zero (it needs the probe phase profile to deconvolve).
2. **`parallax_flip_phase=False`** — leave the parallax phase un-flipped.

In [ ]:
from quantem.diffractive_imaging import DirectPtychography

direct = DirectPtychography.from_dataset4d(
    dset,
    energy=meta["voltage_kV"] * 1e3,                          # 300 kV -> 300000 eV
    semiangle_cutoff=meta["probe_semiangle_mrad"] * 1e-3,     # 30 mrad -> 0.030 rad
    rotation_angle=None,                                       # auto-estimate
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=True,
)
print("DirectPtychography built")

In [ ]:
import time

# Operator's calibrated aberrations for this gold dataset.
# Source: /home/owner/ssd/data/bob/20260408_gold_4dstem_512_ssb/calibration.json
ABER = {"C10": -51.6, "C12": 5.2, "phi12": 0.14}

KERNELS = ["parallax", "ssb", "icom"]
phases = {}
for k in KERNELS:
    t0 = time.time()
    direct.reconstruct(
        deconvolution_kernel=k,
        override_aberration_coefs=ABER,
        parallax_flip_phase=False,
        verbose=False,
    )
    phases[k] = direct.corrected_bf.detach().cpu().numpy()
    print(f"  {k:>9}: {time.time()-t0:.2f}s  range [{phases[k].min():.3f}, {phases[k].max():.3f}]")

## Step 6 — All three kernels side by side

`link_contrast=False` so every kernel gets its own min/max (the SSB output is
~3 orders of magnitude smaller than ICOM).

In [ ]:
quantem.widget.Show2D(
    [phases["parallax"], phases["ssb"], phases["icom"]],
    labels=["parallax", "SSB", "ICOM"],
    sampling=meta["sampling"][:2], units=meta["units"][:2],
    cmap="gray",
    link_contrast=False,
)

## Step 7 — Phase retrieval vs classic imaging

The workshop punchline.

- **BF / DF**: intensity contrast from inside / outside the BF disk. Limited
  by probe size; atomic-lattice fringes mostly washed out.
- **|CoM|**: first-moment deflection per scan position; better than BF/DF.
- **parallax / SSB**: full diffraction pattern deconvolved against the probe
  transfer function. Sharper contrast at the same dose.

Each panel in its own `Show2D` widget (contrast NOT linked across panels —
they live on very different scales).

In [ ]:
quantem.widget.Show2D(bf, title="BF — intensity inside disk", sampling=meta["sampling"][:2], units=meta["units"][:2], cmap="gray")

In [ ]:
quantem.widget.Show2D(df, title="DF — intensity outside disk", sampling=meta["sampling"][:2], units=meta["units"][:2], cmap="gray")

In [ ]:
quantem.widget.Show2D(com_mag.cpu().numpy(), title="|CoM| — first-moment magnitude", sampling=meta["sampling"][:2], units=meta["units"][:2], cmap="magma")

In [ ]:
quantem.widget.Show2D(phases["parallax"], title="parallax — phase retrieval", sampling=meta["sampling"][:2], units=meta["units"][:2], cmap="gray")

In [ ]:
quantem.widget.Show2D(phases["ssb"], title="SSB — phase retrieval", sampling=meta["sampling"][:2], units=meta["units"][:2], cmap="gray")

## What you just did

1. Loaded real 4D-STEM gold from Hugging Face → torch GPU + numpy `Dataset4dstem`.
2. Browsed it with `Show4DSTEM`.
3. BF, DF: inline torch on GPU + separate `Show2D` widgets (independent contrast).
4. DPC: upstream `CenterOfMassOriginModel.from_dataset(..., device="cuda").calculate_origin()` — torch on GPU.
5. Built `DirectPtychography` once, swept three deconvolution kernels (parallax,
   SSB, ICOM) using the operator's calibrated aberrations.
6. Compared all five modalities — each in its own widget.

| Method | What it uses | Result on this dataset |
|---|---|---|
| BF, DF | counts inside / outside the BF disk | smooth intensity, low contrast |
| DPC (`|CoM|`) | first moment per CBED | first-order field deflection |
| parallax, SSB, ICOM | full CBED at every scan position | recovers atomic-lattice phase |

The takeaway: phase retrieval recovers contrast + resolution that BF/DF can
not physically access, at the same dose.

## Try next

- Swap to `gold_512_npy_bin4` for a 4× finer detector.
- Use `direct.optimize_hyperparameters(...)` with `OptimizationParameter` to FIT
  the aberrations from data instead of using the operator value (upstream Optuna
  workflow; currently has an open issue, working manual override above).
- v2 will add iterative ptychography (`PtychoLite`) for the highest-resolution phase.